In [1]:
import pathlib
import pickle

import folium
import mcr_py.helper_functions
import mcr_py.mcr.data
import mcr_py.mcr.path
import mcr_py.mcr5.labels
import mcr_py.minute_city.minute_city
import mcr_py.utils.strtime
import numpy as np
import pandas as pd
import polars as pl
from mcr_py.mcr.path import GTFSPath, Path, PathType
from mcr_py.utils.logger import setup

setup("INFO")

In [2]:
city_name = "cologne"
date = "20250926"

In [3]:
data_directory = pathlib.Path("../data/")
base_directory = data_directory / date
cache_path = base_directory / "cache/"
osm_path = base_directory / "osm_raw"
geometa_path = base_directory / f"cache/{city_name}_geometa.json"
mcr5_output_path = base_directory / f"mcr5_results/{city_name}_reduced_paths"
mcr5_output_path_comp = base_directory / f"mcr5_results/{city_name}"
gtfs_clean_dir = base_directory / f"gtfs_clean/{city_name}/"
gtfs_clean_struct = gtfs_clean_dir / "structs.pkl"
gtfs_clean_stops = gtfs_clean_dir / "stops.parquet"
geo_meta, geo_data = mcr_py.helper_functions.load_auxiliary_classes(
    geo_meta_path=geometa_path,
    city_id="Koeln",
    osm_path=osm_path,
    cache_path=cache_path,
)

[17:08:53] INFO     Loading OSM walking                               ]8;id=339879;file:///home/ppeter/repo/mcr-py/python/mcr_py/mcr/data.py\data.py]8;;\:]8;id=154340;file:///home/ppeter/repo/mcr-py/python/mcr_py/mcr/data.py#63\63]8;;\
[17:08:57] INFO     Loading OSM walking done (3.98 seconds)           ]8;id=41046;file:///home/ppeter/repo/mcr-py/python/mcr_py/mcr/data.py\data.py]8;;\:]8;id=684559;file:///home/ppeter/repo/mcr-py/python/mcr_py/mcr/data.py#63\63]8;;\
           INFO     Loading OSM POIs                                  ]8;id=90691;file:///home/ppeter/repo/mcr-py/python/mcr_py/mcr/data.py\data.py]8;;\:]8;id=458310;file:///home/ppeter/repo/mcr-py/python/mcr_py/mcr/data.py#70\70]8;;\
           INFO     Loading OSM POIs done (0.02 seconds)              ]8;id=317057;file:///home/ppeter/repo/mcr-py/python/mcr_py/mcr/data.py\data.py]8;;\:]8;id=755900;file:///home/ppeter/repo/mcr-py/python/mcr_py/mcr/data.py#70\70]8;;\
           INFO     Loadin

In [4]:
with open(mcr5_output_path / "walking" / "891fa199c77ffff.pkl", "rb") as f:
    hex = pickle.load(f)

In [5]:
comp = pl.read_ipc(mcr5_output_path_comp / "bicycle" / "891fa199c77ffff.feather").join(
    geo_data.pois.select("nearest_osm_node", "poi_type", "lat", "long"),
    how="left",
    left_on="osm_node_id",
    right_on="nearest_osm_node",
)

Could not memory_map compressed IPC file, defaulting to normal read. Toggle off 'memory_map' to silence this warning.


In [6]:
labels = pd.DataFrame(
    [
        (label.node_id, label.values[0], label.values[1], n_transfers, label)
        for n_transfers, bags in hex["bags_i"].items()
        for bag in bags.values()
        for label in bag
    ],
    columns=["osm_node_id", "time", "cost", "n_transfers", "label"],
)

In [7]:
labels = labels.merge(
    geo_data.pois.select(
        pl.col("nearest_osm_node").alias("osm_node_id").cast(pl.Int64),
        pl.col("lat").alias("poi_lat"),
        pl.col("long").alias("poi_long"),
        "poi_type",
    ).to_pandas(),
    how="left",
    on="osm_node_id",
)

In [8]:
labels["duplicate"] = labels.duplicated(subset=["osm_node_id", "time", "cost"], keep=False)

In [9]:
labels[labels["duplicate"]].sort_values(by=["osm_node_id", "time"])

,osm_node_id,time,cost,n_transfers,label,poi_lat,poi_long,poi_type,duplicate
189,359936,289696,0,0,"IntermediateLabel(values=[289696, 0], hidden_v...",50.948703,6.919099,Grocery,True
190,359936,289696,0,0,"IntermediateLabel(values=[289696, 0], hidden_v...",50.948736,6.919010,Health,True
642,360012,291241,0,0,"IntermediateLabel(values=[291241, 0], hidden_v...",50.951833,6.913937,Sustenance,True
643,360012,291241,0,0,"IntermediateLabel(values=[291241, 0], hidden_v...",50.951789,6.914002,Sustenance,True
219,151024266,290810,0,0,"IntermediateLabel(values=[290810, 0], hidden_v...",50.947913,6.919489,Sustenance,True
...,...,...,...,...,...,...,...,...,...
855,11372823688,289291,0,0,"IntermediateLabel(values=[289291, 0], hidden_v...",50.948800,6.918367,Shops,True
504,11372823690,288608,0,0,"IntermediateLabel(values=[288608, 0], hidden_v...",50.949386,6.917702,Banks,True
505,11372823690,288608,0,0,"IntermediateLabel(values=[288608, 0], hidden_v...",50.949538,6.917459,Shops,True
506,11372823690,288608,0,0,"IntermediateLabel(values=[288608, 0], hidden_v...",50.949495,6.917533,Sustenance,True


In [10]:
nodes = geo_data.osm_nodes.with_columns(pl.col("osm_id").alias("id")).to_pandas()

In [11]:
path_manager = hex["path_manager"]

In [12]:
from mcr_py.mcr.data import NetworkType

translator_map = {
    PathType.WALKING: dict(
        geo_data.osm_nodes.select(pl.col("rx_node_id").alias("osm"), "osm_id").rows()
    ),
    PathType.CYCLING_WALKING: dict(
        pl.concat(
            [
                geo_data.osm_nodes.select(
                    pl.lit("W").alias("osm_id") + pl.col("osm_id").cast(pl.String)
                ),
                geo_data.additional_networks[NetworkType.CYCLING][0].select(
                    pl.lit("D").alias("osm_id") + pl.col("osm_id").cast(pl.String)
                ),
            ],
            how="diagonal",
        )
        .with_row_index()
        .rows()
    ),
    PathType.DRIVING_WALKING: dict(
        pl.concat(
            [
                geo_data.osm_nodes.select(
                    pl.lit("W").alias("osm_id") + pl.col("osm_id").cast(pl.String)
                ),
                geo_data.additional_networks[NetworkType.DRIVING][0].select(
                    pl.lit("D").alias("osm_id") + pl.col("osm_id").cast(pl.String)
                ),
            ],
            how="diagonal",
        )
        .with_row_index()
        .rows()
    ),
    PathType.PUBLIC_TRANSPORT: None,
}

In [13]:
def format_meta(meta, previous_meta, start_time):
    values = meta["values"]
    arrival_time = values[0]
    cost = values[1]

    if previous_meta:
        previous_values = previous_meta["values"]
        previous_arrival_time = previous_values[0]
        previous_cost = previous_values[1]

        arrival_time -= previous_arrival_time
        cost -= previous_cost
    else:
        arrival_time -= start_time

    return f"{mcr_py.utils.strtime.seconds_to_str_time(arrival_time, 10)} ({cost})"

In [14]:
labels.poi_type.unique()

array([nan, 'Shops', 'Sustenance', 'Grocery', 'Health', 'Banks', 'Parks',
       'Education'], dtype=object)

In [15]:
color_map = {
    "Shops": "orange",
    "Grocery": "red",
    "Parks": "green",
    "Education": "blue",
    "Banks": "violet",
    "Health": "darkgreen",
    "Sustenance": "yellow",
}

mode_color = {
    PathType.WALKING: "grey",
    PathType.CYCLING_WALKING: "blue",
    PathType.DRIVING_WALKING: "violet",
    PathType.PUBLIC_TRANSPORT: "green",
}

In [ ]:
import plotly.graph_objects as go
from mcr_py.mcr.label import IntermediateLabel

nodes_by_id = nodes.set_index("id", drop=False)
fig = go.Figure()

wrote_walking = False
wrote_node = False
wrote_poi = {
    "Shops": False,
    "Grocery": False,
    "Parks": False,
    "Education": False,
    "Banks": False,
    "Health": False,
    "Sustenance": False,
}
for row in labels.itertuples():
    label: IntermediateLabel = row.label  # pyright: ignore[reportAssignmentType]
    end_node_id = row.osm_node_id
    end_node = nodes_by_id.loc[end_node_id]

    paths = mcr_py.mcr.path.reconstruct_and_translate_path_for_label(
        path_manager.paths, label, translator_map
    )
    for i, path in enumerate(paths):
        if isinstance(path, Path):
            if path.path == []:
                continue
            if path.path_type == PathType.WALKING:
                walking_path_nodes = [nodes_by_id.loc[node_id] for node_id in path.path]
                path_lat = [node.lat for node in walking_path_nodes]
                path_lon = [node.long for node in walking_path_nodes]
                if i + 1 == len(paths):
                    path_lat.append(end_node.lat)  # type: ignore
                    path_lon.append(end_node.long)  # type: ignore
                else:
                    path_lat.append(nodes_by_id.loc[int(paths[i + 1].path[0][1:])].lat)
                    path_lon.append(nodes_by_id.loc[int(paths[i + 1].path[0][1:])].long)
                fig.add_trace(
                    go.Scattermap(
                        lon=path_lon,
                        lat=path_lat,
                        mode="lines",
                        marker={"size": 1, "color": "rgb(96,96,96)"},
                        legendgroup="Walking",
                        name="Walking",
                        showlegend=not wrote_walking,
                    )
                )
                if not wrote_walking:
                    wrote_walking = not wrote_walking
            if path.path_type in [PathType.DRIVING_WALKING, PathType.CYCLING_WALKING]:
                path_nodes = [
                    nodes_by_id.loc[int(node_id[1:])]  # pyright: ignore[reportIndexIssue]
                    for node_id in path.path
                    if node_id[0] == "D"  # pyright: ignore[reportIndexIssue]
                ]
                path_lat = [node.lat for node in path_nodes]
                path_lon = [node.long for node in path_nodes]
                fig.add_trace(
                    go.Scattermap(
                        lon=path_lon,
                        lat=path_lat,
                        mode="lines",
                        line={"dash": "dot"},
                        marker={"size": 1, "color": "rgb(51,51,255)"},
                        legendgroup="Bicycle",
                    )
                )
    fig.add_trace(
        go.Scattermap(
            lon=[end_node.long],
            lat=[end_node.lat],
            mode="markers",
            marker={"size": 10, "color": "black"},
            name="Node",
            legendgroup="Nodes",
            showlegend=not wrote_node,
        )
    )
    if not wrote_node:
        wrote_node = not wrote_node
    if row.poi_type is not np.nan:
        fig.add_trace(
            go.Scattermap(
                lon=[row.poi_long],
                lat=[row.poi_lat],
                mode="markers",
                marker={"size": 10, "color": color_map[row.poi_type]},  # pyright: ignore[reportArgumentType]
                name=row.poi_type,
                legendgroup=row.poi_type,
                showlegend=not wrote_poi[row.poi_type],  # pyright: ignore[reportArgumentType]
            )
        )
        if not wrote_poi[row.poi_type]:  # pyright: ignore[reportArgumentType]
            wrote_poi[row.poi_type] = not wrote_poi[row.poi_type]  # pyright: ignore[reportArgumentType]
        fig.add_trace(
            go.Scattermap(
                lon=[row.poi_long, end_node.long],
                lat=[row.poi_lat, end_node.lat],
                mode="lines",
                marker={"size": 10, "color": color_map[row.poi_type]},  # pyright: ignore[reportArgumentType]
                showlegend=False,
            )
        )

fig.update_layout(
    map={
        "style": "basic",
        "zoom": 15,  # street level
        "center": {"lat": 50.948884, "lon": 6.917342},
    },
    margin={"r": 0, "t": 0, "l": 0, "b": 0},
    height=800,
    width=1400,
)
fig.write_image("map.png", scale=2)

[17:09:14] INFO     Chromium init'ed with kwargs {}              ]8;id=167285;file:///home/ppeter/repo/mcr-py/.venv/lib/python3.13/site-packages/choreographer/browsers/chromium.py\chromium.py]8;;\:]8;id=151480;file:///home/ppeter/repo/mcr-py/.venv/lib/python3.13/site-packages/choreographer/browsers/chromium.py#182\182]8;;\
           INFO     Found chromium path: /usr/bin/chromium       ]8;id=353571;file:///home/ppeter/repo/mcr-py/.venv/lib/python3.13/site-packages/choreographer/browsers/chromium.py\chromium.py]8;;\:]8;id=474392;file:///home/ppeter/repo/mcr-py/.venv/lib/python3.13/site-packages/choreographer/browsers/chromium.py#209\209]8;;\
           INFO     Temp directory created: /tmp/tmpdu9hpugx.     ]8;id=332793;file:///home/ppeter/repo/mcr-py/.venv/lib/python3.13/site-packages/choreographer/utils/_tmpfile.py\_tmpfile.py]8;;\:]8;id=62535;file:///home/ppeter/repo/mcr-py/.venv/lib/python3.13/site-packages/choreographer/utils/_tmpfile.py#80\80]8;;\
         

TimeoutError: 

In [ ]:
from mcr_py.mcr.label import IntermediateLabel

toloop = labels

# stops_by_id = stops_df.set_index("stop_id")
sample_label = labels.iloc[0]
sample_node_id = sample_label.osm_node_id
nodes_by_id = nodes.set_index("id", drop=False)
sample_node = nodes_by_id.loc[sample_node_id]
start_time = 288000

m = folium.Map(location=[50.948884, 6.917342], zoom_start=17)


for row in toloop.itertuples():
    label: IntermediateLabel = row.label  # pyright: ignore[reportAssignmentType]
    end_node_id = row.osm_node_id
    end_node = nodes_by_id.loc[end_node_id]

    paths = mcr_py.mcr.path.reconstruct_and_translate_path_for_label(
        path_manager.paths, label, translator_map
    )
    for i, path in enumerate(paths):
        if isinstance(path, Path):
            if path.path == []:
                continue
            if path.path_type == PathType.WALKING:
                walking_path_nodes = [nodes_by_id.loc[node_id] for node_id in path.path]
                path_lat_lon = [(node.lat, node.long) for node in walking_path_nodes]
                if i + 1 == len(paths):
                    path_lat_lon.append([end_node.lat, end_node.long])
                else:
                    path_lat_lon.append(
                        (
                            nodes_by_id.loc[int(paths[i + 1].path[0][1:])].lat,
                            nodes_by_id.loc[int(paths[i + 1].path[0][1:])].long,
                        )
                    )
                last_node = path_lat_lon[-1]

                previous_meta = paths[i - 1].meta if i > 0 else None
                meta = format_meta(path.meta, previous_meta, start_time)
                if path_lat_lon != []:
                    folium.PolyLine(
                        [*path_lat_lon],
                        color="grey",
                        weight=2,
                        popup=str(meta),
                    ).add_to(m)
            if path.path_type in [PathType.DRIVING_WALKING, PathType.CYCLING_WALKING]:
                path_nodes = [
                    nodes_by_id.loc[int(node_id[1:])]
                    for node_id in path.path
                    if node_id[0] == "D"
                ]
                path_lat_lon = [(node.lat, node.long) for node in path_nodes]
                previous_meta = paths[i - 1].meta if i > 0 else None
                meta = format_meta(path.meta, previous_meta, start_time)
                if path_lat_lon != []:
                    folium.PolyLine(
                        path_lat_lon,
                        color=mode_color[path.path_type],
                        weight=2,
                        popup=str(meta),
                        dash_array=10,
                    ).add_to(m)
        elif isinstance(path, GTFSPath):
            print("Impossible")
            start_stop_id = path.start_stop_id
            end_stop_id = path.end_stop_id
            start_stop = stops_by_id.loc[start_stop_id]
            end_stop = stops_by_id.loc[end_stop_id]
            trip = path.trip_id
            if len(trip) >= 10:
                trip = trip[:10] + "..."

            previous_meta = paths[i - 1].meta if i > 0 else None
            line_msg = f"Trip: {trip}\n---\n {format_meta(path.meta, previous_meta)}"

            path_lat_lon = [
                (float(start_stop.stop_lat), float(start_stop.stop_lon)),
                (float(end_stop.stop_lat), float(end_stop.stop_lon)),
            ]
            folium.PolyLine(
                path_lat_lon,
                color="green",
                weight=2,
                popup=line_msg,
            ).add_to(m)

            folium.CircleMarker(
                location=[float(start_stop.stop_lat), float(start_stop.stop_lon)],
                popup=f"Start: {start_stop.stop_name}",
                color="green",
                radius=3,
            ).add_to(m)
            folium.CircleMarker(
                location=[float(end_stop.stop_lat), float(end_stop.stop_lon)],
                popup=f"End: {end_stop.stop_name}",
                color="green",
                radius=3,
            ).add_to(m)
        else:
            raise Exception("Unknown path type")

    folium.CircleMarker(
        location=[end_node.lat, end_node.long],  # pyright: ignore[reportArgumentType]
        popup=f"End: {end_node_id}",
        color="black",
        radius=2,
    ).add_to(m)
    if row.poi_type is not np.nan:
        folium.CircleMarker(
            location=[row.poi_lat, row.poi_long],  # pyright: ignore[reportArgumentType]
            popup=f"POI: {row.poi_type}",
            color=color_map[row.poi_type],
            radius=4,
        ).add_to(m)
        folium.PolyLine(
            [(row.poi_lat, row.poi_long), (end_node.lat, end_node.long)],
            color=color_map[row.poi_type],
            weight=2,
            popup=f"POI: {row.poi_type}",
        ).add_to(m)


folium.Marker(location=[50.948884, 6.917342], icon=folium.Icon("green"), popup="Start").add_to(
    m
)
m

In [ ]:
for path in path_manager.paths.values():
    if path.path_type == PathType.PUBLIC_TRANSPORT:
        print(path)

In [ ]:
for path in path_manager.paths.values():
    if path.path_type == PathType.PUBLIC_TRANSPORT:
        print(path)